# BAMv3 실험: 노이즈→노이즈 학습

**Ultra-low SNR 환경을 위한 Robust Feature Learning**
- Self-supervised 학습 방식
- 다양한 SNR에서 일관된 패턴 학습
- 병목 효과로 랜덤 노이즈 제거


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa, MultiBAMv3
import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)


In [ ]:
# LoRa 파라미터 설정
sf = 9
bw = 250_000  # 250 kHz
OSF = 4
fs = int(bw * OSF)  # 1 MHz

print(f"SF = {sf}, BW = {bw/1e3:.1f} kHz, fs = {fs/1e6:.3f} MHz")

# LoRa 인스턴스 생성 (on-the-fly 노이즈 추가용)
lora = LoRa(sf, bw)


In [ ]:
########################################
##  1. 클린 IQ 데이터 로드            ##
########################################

base_dir = "dataset_v3_sf9_bw250k"
iq_dir = os.path.join(base_dir, "clean_iq")

print("Loading clean IQ data...")
iq_files = sorted(glob.glob(os.path.join(iq_dir, "*.npy")))
print(f"Found {len(iq_files)} clean IQ files")

# 클린 IQ 로드
iq_data_list = []
for f in iq_files:
    x_clean = np.load(f)  # complex IQ
    iq_data_list.append(x_clean)

X_clean_iq = np.array(iq_data_list)
print(f"Clean IQ shape: {X_clean_iq.shape}, dtype: {X_clean_iq.dtype}")


In [ ]:
########################################
##  2. On-the-fly 노이즈 추가        ##
########################################

print("\nAdding noise on-the-fly...")

# SNR 범위 설정 (Ultra-low SNR 포함)
SNR_MIN = -30
SNR_MAX = 15

# 각 클린 IQ에 랜덤 SNR로 노이즈 추가
X_noisy_iq_list = []
snr_list = []

for x_clean_iq in X_clean_iq:
    # 랜덤 SNR 선택
    snr = np.random.randint(SNR_MIN, SNR_MAX + 1)
    snr_list.append(snr)
    
    # 노이즈 추가
    x_noisy_iq = lora.awgn_iq(x_clean_iq, snr)
    X_noisy_iq_list.append(x_noisy_iq)

X_noisy_iq = np.array(X_noisy_iq_list)
print(f"Noisy IQ shape: {X_noisy_iq.shape}")
print(f"SNR range used: {min(snr_list)} ~ {max(snr_list)} dB")


In [ ]:
########################################
##  3. 노이즈 스펙트로그램 생성       ##
########################################

print("\nGenerating noisy spectrograms...")

X_noisy_spec_list = []
for x_noisy_iq in X_noisy_iq:
    spec = generate_spectrogram(x_noisy_iq, fs)
    X_noisy_spec_list.append(spec)

X_noisy_spec = np.array(X_noisy_spec_list)
print(f"Noisy spectrogram shape: {X_noisy_spec.shape}")

# Flatten to 1D for training
X_noisy_flat = np.array([s.flatten() for s in X_noisy_spec])
print(f"Noisy flat shape: {X_noisy_flat.shape}")


In [ ]:
########################################
##  4. BAMv3 Self-supervised 학습    ##
########################################

# 모델 하이퍼파라미터
input_dim = X_noisy_flat.shape[1]  # 4352 (256 * 17)
hidden_dim = 2048
output_dim = 512

print(f"\n=== Training BAMv3 Model ===")
print(f"Architecture: {input_dim} → {hidden_dim} → {output_dim}")
print(f"Learning Strategy: Noisy → Noisy (Self-supervised)")

# BAMv3 모델 생성
layers = [input_dim, hidden_dim, output_dim]
model = MultiBAMv3(layers_dims=layers, eta=1e-5)

# 학습 플래그
TRAIN = True

if TRAIN:
    print("\n=== Starting Training ===")
    
    # 학습 파라미터
    num_epochs = 10
    batch_size = 64
    
    # BAMv3 학습 (노이즈 → 노이즈)
    layer_losses = model.train(X_noisy_flat, num_epochs=num_epochs, batch_size=batch_size)
    
    print("\n=== Training Complete ===")
    
    # Loss 플롯 (각 layer별)
    fig, axes = plt.subplots(1, len(layer_losses), figsize=(15, 4))
    if len(layer_losses) == 1:
        axes = [axes]
    
    for i, losses in enumerate(layer_losses):
        axes[i].plot(losses)
        axes[i].set_xlabel('Batch')
        axes[i].set_ylabel('MSE Loss')
        axes[i].set_title(f'Layer {i+1} Training Loss')
        axes[i].grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("\nTraining skipped (TRAIN=False)")


In [ ]:
########################################
##  5. 모델 저장                     ##
########################################

if TRAIN:
    # 모델 저장 폴더
    weight_folder = "weights_bamv3"
    os.makedirs(weight_folder, exist_ok=True)
    
    # 각 layer의 weight 저장
    for i, bam in enumerate(model.bams):
        weight_path = os.path.join(weight_folder, f"weights_layer_{i}.npy")
        np.save(weight_path, bam.W.cpu().numpy())
        print(f"✅ Layer {i} weights saved to: {weight_path}")
    
    # 아키텍처 정보 저장
    config = {
        'layers': layers,
        'eta': 1e-5,
        'sf': sf,
        'bw': bw,
        'fs': fs
    }
    config_path = os.path.join(weight_folder, "model_config.npy")
    np.save(config_path, config)
    print(f"✅ Config saved to: {config_path}")

print("\n=== How to Load Model ===")
print("""
# 모델 로드 예시:
config = np.load('weights_bamv3/model_config.npy', allow_pickle=True).item()
model = MultiBAMv3(layers_dims=config['layers'], eta=config['eta'])
for i, bam in enumerate(model.bams):
    bam.W = torch.tensor(np.load(f'weights_bamv3/weights_layer_{i}.npy'))
""")


In [ ]:
########################################
##  6. 테스트: 복원 성능 확인        ##
########################################

# 테스트 샘플 선택
test_idx = 0

print(f"\n=== Testing Reconstruction on Sample {test_idx} ===")
print(f"SNR used: {snr_list[test_idx]} dB")

# 입력 준비
x_noisy = X_noisy_flat[test_idx:test_idx+1]  # (1, 4352)

# 모델 예측 (압축 → 복원)
z_compressed = model.compress(x_noisy)
x_reconstructed = model.decompress(z_compressed)

# Reshape back to spectrogram
spec_noisy = x_noisy.reshape(256, 17)
spec_reconstructed = x_reconstructed.reshape(256, 17)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im1 = axes[0].imshow(spec_noisy, aspect='auto', cmap='viridis')
axes[0].set_title(f'Noisy Input (SNR={snr_list[test_idx]} dB)')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Frequency')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(spec_reconstructed, aspect='auto', cmap='viridis')
axes[1].set_title('BAMv3 Reconstructed')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Frequency')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

# MSE 계산
mse = np.mean((spec_reconstructed - spec_noisy) ** 2)
print(f"\nReconstruction MSE: {mse:.6f}")
